# LaneRush — Emergency Vehicle Classifier Training (Google Colab)

This notebook trains a **YOLOv8n-cls** model on the Kaggle Emergency Vehicle Dataset.

- Backbone: Pretrained `yolov8n-cls.pt` (transfer learning)
- Output: `ev_classifier.pt` weights and `metrics.json` (precision, recall, false alarm rate, confusion matrix)

In [ ]:
# 1. Install ultralytics
!pip install -q ultralytics

In [ ]:
# 2. Clone repository or upload files

!mkdir -p data/raw data/ev_cls ml/results ml/runs

### 3. Download the Kaggle Emergency Vehicle Dataset
Upload your `kaggle.json` or download the dataset directly into `data/raw/`.

In [ ]:
# If using Kaggle API:
# !pip install -q kaggle
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c emergency-vehicle-detection -p data/raw/
# !unzip -q data/raw/*.zip -d data/raw/

In [ ]:
# 4. Prepare data: fixed-seed 70/15/15 train/val/test split
# Run prepare_data.py on the raw dataset
!python ml/prepare_data.py data/raw --out_dir data/ev_cls --seed 42

In [ ]:
# 5. Train YOLOv8n-cls classifier
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')
results = model.train(
    data='data/ev_cls',
    epochs=30,
    imgsz=224,
    batch=64,
    seed=42,
    project='ml/runs',
    name='ev_cls_run'
)

In [ ]:
# 6. Evaluate on test split and generate metrics.json
best_weights = 'ml/runs/ev_cls_run/weights/best.pt'
!python ml/evaluate_cls.py {best_weights} data/ev_cls --out_dir ml/results/

In [ ]:
# 7. Download trained weights and metrics.json to commit to repo
from google.colab import files

files.download('ml/results/metrics.json')
files.download(best_weights)